In [1]:
import geoai

In [2]:
from sql import sedona_vectorized_udf
from sql import SedonaUDFType
import numpy as np
from IPython.display import Image, display
from sedona.spark import SedonaContext
import itertools
import os
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from sedona.stats.clustering.dbscan import dbscan
import pyspark.sql.functions as f
from sedona.sql.types import GeometryType

In [ ]:
%%capture
additional_packages = [
    'org.apache.sedona:sedona-spark-3.5_2.12:1.7.2',
    'org.datasyslab:geotools-wrapper:1.7.2-28.5',
    'org.apache.hadoop:hadoop-aws:3.3.4',
    'org.apache.hadoop:hadoop-client-api:3.3.4',
    'org.apache.hadoop:hadoop-common:3.3.4',
]

config_params = {
    "spark.jars.packages": ",".join(additional_packages),
    "spark.hadoop.fs.s3a.aws.credentials.provider": "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
}

if os.environ.get("SEDONA_COPY_MINIO") == "true":
    config_params = {
        **config_params,
        **{
            "spark.hadoop.fs.s3a.access.key": "sedona",
            "spark.hadoop.fs.s3a.secret.key": "sedona_password",
            "spark.hadoop.fs.s3a.endpoint": "http://minio:9000",
            "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
            "spark.hadoop.fs.s3a.path.style.access": "true",
            "spark.driver.memory": "2G",
            "spark.executor.memory": "2G"
        }
    }

bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

for key, value in config_params.items():
    config = config.config(key, value)

sedona = SedonaContext.create(config.getOrCreate())
sedona.sparkContext.setLogLevel("ERROR")

In [9]:
from shapely.geometry import Point
import shapely.geometry.base as b
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader,Dataset

from torchvision.models.detection import maskrcnn_resnet50_fpn
import torch

image_mean = [0.485, 0.456, 0.406]
image_std = [0.229, 0.224, 0.225]

# Create model with explicit normalization parameters
model = maskrcnn_resnet50_fpn(
    weights=None,
    progress=False,
    num_classes=2,  # Background + object
    weights_backbone=None,
    # These parameters ensure consistent normalization
    image_mean=image_mean,
    image_std=image_std,
)

In [10]:
!wget https://huggingface.co/giswqs/geoai/resolve/main/building_footprints_usa.pth

--2025-06-19 21:40:34--  https://huggingface.co/giswqs/geoai/resolve/main/building_footprints_usa.pth
Resolving huggingface.co (huggingface.co)... 108.138.51.21, 108.138.51.26, 108.138.51.41, ...
Connecting to huggingface.co (huggingface.co)|108.138.51.21|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.hf.co/repos/71/4f/714f3e83e1942e1e338a873c6937f14cb7d9d3372dd2003495c01190dca64d3a/3aea5d0da7803be31ff4e2e1e5ac747f15cab2a13d6bb1a9319b6a6ba8bc7bca?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27building_footprints_usa.pth%3B+filename%3D%22building_footprints_usa.pth%22%3B&Expires=1750372834&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc1MDM3MjgzNH19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zLzcxLzRmLzcxNGYzZTgzZTE5NDJlMWUzMzhhODczYzY5MzdmMTRjYjdkOWQzMzcyZGQyMDAzNDk1YzAxMTkwZGNhNjRkM2EvM2FlYTVkMGRhNzgwM2JlMzFmZjRlMmUxZTVhYzc0N2YxNWNhYjJhMTNkNmJiMWE5MzE5YjZhNmJhO

In [11]:
%%capture
import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model.to(device)
model_path = "building_footprints_usa.pth"
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

# prediction function

In [12]:
from shapely.geometry import shape
from shapely.ops import unary_union
from rasterio import features

def predict(img):
    # our data is 4 bands, we need to take only RGB
    np_array = img.read()[:3,::]

    # we need to convert the numpy array to internal tensor model in torch
    img_normalized = torch.from_numpy(np_array).float()

    # if the image maximum value is greater than 1 we need to normalize the
    # image
    if img_normalized.max() > 1:
        img_normalized = img_normalized / 255.0

    # context manager to run the torch models
    with torch.no_grad():
        # we call the model on the tensor representing the input image
        output = model([img_normalized])[0]

        # here we store all unique buildings, then we will use it to create
        # GeometryCollection object
        result_shapes = []

        # we iterate through all unique masks we get from the prediction, each          
        # mask represents the detected building
        for m in result["masks"]:
            # the result is torch tensor, we need to use numpy in the rasterio,
            # raster to vector convertions
            m_numpy = m.numpy()

            # each prediction has probability assigned, we need trustworthy
            # result so we create mask with prediction value greater than 0.7
            mask = m_numpy > 0.7

            # this function for the mask and input array create the iterator 
            # with polygon representing each pixel we also pass affine
            # transformation object to make sure our geometry is properly
            # georeferenced
            shapes = features.shapes(
                m_numpy,
                mask=mask,
                transform=img.transform
            )

            # here we store all the pixel polygons which we will use to union
            # them together
            polygons = []
            for s in shapes:
                # each s[0] is geojson object we need to convert it to the
                # shapely base geometry
                polygons.append(shape(s[0]))

            # merging pixels to one polygon/multipolygon geometry
            merged = unary_union(polygons)
            
            result_shapes.append(unary_union(polygons))

        # we return GeometryCollection object
        return GeometryCollection(result_shapes)

In [18]:
import uuid
from shapely.geometry import Point, LineString, Polygon, GeometryCollection

@sedona_vectorized_udf(
    udf_type=SedonaUDFType.GEO_SERIES,
    return_type=GeometryType()
)
def extract_buildings(series) -> b.BaseGeometry:
    df = series.apply(lambda x: predict(deserialize(x).as_rasterio()))
    
    return df

In [14]:
images = sedona.read.format("binaryFile").load(f"s3a://{bucket_name}/source_data/buildings_raster").limit(2)

images.cache().count()

2

In [19]:
buildings_predicted = images.\
    selectExpr("RS_FromGeoTiff(content) AS rast"). \
    select(vectorized_numeric_to_geom(f.col("rast")).alias("geom"))

In [ ]:
images_predicted.show()

In [ ]:
buildings_predicted.\
    selectExpr("EXPLODE(ST_Dump(geom)) AS geometry").\
    show()